<h1>Result_5</h1>

In [ ]:
import anndata as ad
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import warnings
import os  
import loompy
import gzip
import shutil
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import mannwhitneyu

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(dpi=50, dpi_save=300, format='png', 
                             frameon=False, transparent=True, fontsize=10, figsize=(4, 4))

warnings.simplefilter(action='ignore', category=FutureWarning)

plt.rcParams["image.aspect"] = "equal"
plt.rcParams["figure.figsize"] = ([4, 4])  
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.grid'] = False

colorrs = ["#4DBBD5", "#00A087", "#E64B35","#3C5488", "#F39B7F", "#8491B4",
        "#91D1C2",  "#B9C984",  "#9ACBDE", "#F494BE", "#EDCAE0", 
        "#C8CADF", "#F47892", "#F6A395",  "#C9AFA2", "#ABADC5", "#AEB9AC", 
        "#4b6aa8", "#3ca0cf", "#c376a7", "#ad98c3", "#cea5c7",
        "#53738c", "#a5a9b0", "#a78982", "#696a6c", "#92699e",
        "#d69971", "#df5734", "#6c408e", "#ac6894", "#d4c2db",
        "#537eb7", "#83ab8e", "#ece399", "#405993", "#cc7f73",
        "#b95055", "#d5bb72", "#bc9a7f", "#e0cfda", "#d8a0c0",
        "#d69a55", "#64a776", "#cbdaa9",
        "#efd2c9", "#da6f6d", "#ebb1a4", "#a44e89", "#a9c2cb",
        "#b85292", "#6d6fa0", "#8d689d", "#c8c7e1", "#d25774",
        "#c49abc", "#927c9a", "#3674a2", "#9f8d89", "#72567a",
        "#63a3b8", "#c4daec", "#61bada", "#b7deea", "#e29eaf",
        "#4490c4", "#e6e2a3",  "#c4612f", "#9a70a8",
        "#76a2be", "#408444", "#c6adb0", "#9d3b62", "#2d3462"]

In [ ]:
highlight_celltypes = ["B_01_Naive_TCL1A_IGHD","B_03_Memory_CD27","B_02_iMemory_IGHD_CD27","B_04_Plasma_IGHA1_IGHG1"]
highlight_colors = ["#4DBBD5","#00A087","#E64B35","#3C5488"]
highlight_palette = dict(zip(highlight_celltypes, highlight_colors))
palette = {ct:highlight_palette.get(ct,"lightgray") for ct in adata.obs["celltype"].astype("category").cat.categories}

fig, ax = plt.subplots(figsize=(4.3,2.5), dpi=300)
sc.pl.umap(adata, color="celltype", size=0.5, palette=palette, ax=ax, show=False, frameon=True, legend_loc=None)

handles = [Line2D([0],[0],marker="o",color="w",markerfacecolor=highlight_palette[ct],markersize=6,markeredgecolor="none") for ct in highlight_celltypes]
ax.legend(handles, highlight_celltypes, bbox_to_anchor=(1.05,0.5), loc="center left", frameon=False, fontsize=8)

ax.spines[["top","right"]].set_visible(False)
ax.set_xlabel("UMAP1", fontsize=8)
ax.set_ylabel("UMAP2", fontsize=8)
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)

fig.tight_layout()
fig.savefig("Fig.5/B_cell_subsets_UMAP.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
raw_adata = adata.raw.to_adata()
Bcell = raw_adata[raw_adata.obs['celltype_major2']=='B cells'].copy()
sc.pl.umap(Bcell)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib.colors import LinearSegmentedColormap

markers = ["CD79A","CD79B","TCL1A","IGHD","CD27","IGHA1","IGHG1"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#4DBBD5","white","#E64B35"], N=30)

with plt.rc_context({"axes.linewidth":0.5,"xtick.major.width":0.5,"ytick.major.width":0.5}):
    fig = sc.pl.stacked_violin(Bcell, markers, groupby="celltype", cmap=cmap, figsize=(4,2),
                               linewidth=0.5, colorbar_title="Expression", return_fig=True)

fig.savefig("Fig.S13/Bcell_stacked_violin_markers.pdf", dpi=300, bbox_inches="tight")
plt.show()

<h1>plasma</h1>

In [ ]:
plasma_cells = Bcell[Bcell.obs['celltype'] == 'B_04_Plasma_IGHA1_IGHG1']

In [ ]:
igh_genes = ["IGHG1","IGHG2","IGHG3","IGHG4","IGHA1","IGHA2","IGHM"]
colors = ["#4DBBD5","#00A087","#E64B35","#3C5488","#F39B7F","#8491B4","#91D1C2"]

genes = [g for g in igh_genes if g in plasma_cells.var_names]
expr = plasma_cells[:, genes].X
if hasattr(expr, "toarray"): expr = expr.toarray()

max_expr = expr.max(axis=1)
dominant = np.array(genes)[expr.argmax(axis=1)]
ig_summary = pd.Series(dominant[max_expr>0]).value_counts().reindex(genes, fill_value=0)
ig_summary = ig_summary[ig_summary>0] / ig_summary.sum() * 100

fig, ax = plt.subplots(figsize=(4,4), dpi=100)
radius = 0.9
wedges, _ = ax.pie(ig_summary, colors=colors[:len(ig_summary)], startangle=90,
                   wedgeprops={"edgecolor":"white","linewidth":0.5}, radius=radius)

kw = {"arrowprops":{"arrowstyle":"-","color":"black","lw":0.5},
      "bbox":{"boxstyle":"round,pad=0.3","fc":"white","ec":"black","lw":0.3}, "va":"center"}

for wedge, gene, pct in zip(wedges, ig_summary.index, ig_summary):
    ang = (wedge.theta1+wedge.theta2)/2
    x,y = radius*np.cos(np.deg2rad(ang)), radius*np.sin(np.deg2rad(ang))
    r = 1.1 if pct>5 else 1.2
    kw["arrowprops"]["connectionstyle"] = f"angle,angleA=0,angleB={ang}"
    ax.annotate(f"{gene}\n{pct:.1f}%", xy=(x,y), xytext=(r*np.sign(x),r*np.sin(np.deg2rad(ang))),
                ha="left" if x>0 else "right", fontsize=10, **kw)

ax.set_title("Ig classes of plasma cells", fontsize=12, pad=20)
ax.axis("equal")
plt.tight_layout()
plt.savefig("Fig.5/Plasma_Cells_Dominant_IGH.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
markers = ["IGHG1","IGHG2","IGHG3","IGHG4","IGHA1","IGHA2","IGHM"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(plasma_cells, var_names=markers, groupby="Condition", standard_scale="var",
                   categories_order=["HC","MKPP","SKPP"], figsize=(3,3))
dp.swap_axes().style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True,
                     smallest_dot=20, dot_max=0.4).legend(size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=10, rotation=45)
ax.tick_params(axis="y", labelsize=10)
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.5/IGH_expression.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
genes = ["PRDM1","XBP1","IRF4"]
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("MKPP","SKPP"),("HC","SKPP")]
x = np.arange(3)

expr = plasma_cells.raw[:,genes].X if plasma_cells.raw is not None else plasma_cells[:,genes].X
if hasattr(expr,"toarray"): expr = expr.toarray()

cell_df = pd.DataFrame(expr,index=plasma_cells.obs_names,columns=genes)
cell_df[["Sample","Condition"]] = plasma_cells.obs[["Sample","Condition"]]
sample_df = cell_df.groupby(["Sample","Condition"],observed=True)[genes].mean().reset_index()
plot_df = sample_df.melt(id_vars=["Sample","Condition"],value_vars=genes,var_name="Gene",value_name="Expression")

stars_dict = {}
for gene in genes:
    sub = plot_df[plot_df["Gene"]==gene]
    groups = [sub.loc[sub["Condition"]==c,"Expression"] for c in condition_order]
    kw_p = kruskal(*groups).pvalue
    dunn = sp.posthoc_dunn(sub,val_col="Expression",group_col="Condition",p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in pairs]
    stars_dict[gene] = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

fig,axes = plt.subplots(1,len(genes),figsize=(len(genes)*2.5,4),sharey=False)

for ax,gene in zip(axes,genes):
    sub = plot_df[plot_df["Gene"]==gene]
    ymin,ymax = sub["Expression"].agg(["min","max"]); yrange = ymax-ymin if ymax!=ymin else 1
    heights = [ymax+yrange*i for i in [0.15,0.30,0.45]]

    for i,(condition,color) in enumerate(zip(condition_order,colors)):
        data = sub.loc[sub["Condition"]==condition,"Expression"].to_numpy()
        ax.scatter(np.random.normal(x[i]-0.25,0.04,len(data)),data,s=28,color=color,alpha=0.8,edgecolor="white",linewidth=0.5,zorder=3)
        box = ax.boxplot(data,positions=[x[i]-0.05],widths=0.1,patch_artist=True,showfliers=False,zorder=4)
        plt.setp(box["boxes"],facecolor="white",edgecolor=color,linewidth=1.5)
        plt.setp(box["whiskers"]+box["caps"]+box["medians"],color=color,linewidth=1.5)
        if len(data)>=2:
            violin = ax.violinplot(data,positions=[x[i]+0.1],showextrema=False)
            for body in violin["bodies"]:
                body.set(facecolor=color,alpha=0.65,edgecolor="none")
                v = body.get_paths()[0].vertices; v[:,0] = np.clip(v[:,0],x[i]+0.1,np.inf)

    for (a,b),star,h in zip(pairs,stars_dict[gene],heights):
        x1,x2 = condition_order.index(a),condition_order.index(b)
        ax.plot([x1,x2],[h,h],lw=1,color="black")
        ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=12,fontweight="bold" if star!="ns" else "normal")

    ax.set_title(gene,fontsize=12,fontweight="bold",pad=12)
    ax.set_xticks(x,condition_order,rotation=45,ha="center",fontsize=12)
    ax.tick_params(axis="y",labelsize=12)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(1)
    ax.set_ylim(ymin-yrange*0.1,ymax+yrange*0.6)
    ax.grid(False)

axes[0].set_ylabel("Log normalized expression",fontsize=12)
plt.tight_layout()
plt.savefig("Fig.5/Plasma_core_TF_expression_Native_Raincloud.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
plasma_type = "B_04_Plasma_IGHA1_IGHG1"
tfh_type = "CD4T_03_Tfh_ICOS_SLAMF1"

obs = adata.obs[adata.obs["Condition"].isin(["MKPP","SKPP"])].copy()
counts = obs.groupby(["Condition","Sample"], observed=True).agg(
    Plasma_Cells=("celltype", lambda x: (x==plasma_type).sum()),
    Tfh_Cells=("celltype", lambda x: (x==tfh_type).sum()),
    Total_Cells=("celltype","size")
).reset_index()
counts.to_csv("Fig.5/Plasma_Tfh_patient_counts.csv", index=False)

def spearman_bootstrap(df, n_boot=5000, seed=42):
    x,y = df["Plasma_Cells"].to_numpy(float),df["Tfh_Cells"].to_numpy(float)
    rho,p = stats.spearmanr(x,y)
    rng = np.random.default_rng(seed)
    boot = [stats.spearmanr(x[idx],y[idx]).statistic for idx in (rng.integers(0,len(x),len(x)) for _ in range(n_boot))]
    boot = np.asarray(boot); boot = boot[np.isfinite(boot)]
    ci = np.percentile(boot,[2.5,97.5])
    return len(x),rho,ci[0],ci[1],p

results = []
for i,condition in enumerate(["MKPP","SKPP"]):
    sub = counts[counts["Condition"]==condition]
    n,rho,lo,hi,p = spearman_bootstrap(sub,seed=42+i)
    results.append([condition,n,rho,lo,hi,p])

results_df = pd.DataFrame(results,columns=["Condition","n","rho","CI_lower","CI_upper","P_raw"])
results_df["P_FDR"] = multipletests(results_df["P_raw"],method="fdr_bh")[1]
results_df.to_csv("Fig.5/Plasma_Tfh_Spearman_statistics.csv", index=False)

colors = {"MKPP":"#4DBBD5","SKPP":"#E64B35"}
files = {"MKPP":"Figure_5F_MKPP_Plasma_Tfh","SKPP":"Figure_S11B_SKPP_Plasma_Tfh"}
format_p = lambda p: "<0.001" if p<0.001 else f"{p:.3f}"

sns.set_theme(style="ticks")
for condition in ["MKPP","SKPP"]:
    sub = counts[counts["Condition"]==condition]
    res = results_df[results_df["Condition"]==condition].iloc[0]

    fig,ax = plt.subplots(figsize=(4.2,4))
    sns.regplot(data=sub,x="Plasma_Cells",y="Tfh_Cells",ci=95,ax=ax,
                scatter_kws={"s":50,"color":colors[condition],"alpha":0.9,"edgecolors":"black"},
                line_kws={"color":"#E64B35","linewidth":2})

    text = f"$n$ = {int(res['n'])}\n$\\rho$ = {res['rho']:.3f}\n95% CI: {res['CI_lower']:.3f} to {res['CI_upper']:.3f}\n$P$ = {format_p(res['P_raw'])}; FDR = {format_p(res['P_FDR'])}"
    ax.text(0.04,0.96,text,transform=ax.transAxes,ha="left",va="top",fontsize=10)
    ax.set_title("Correlation between Plasma and Tfh",fontsize=13)
    ax.set_xlabel(f"Plasma-cell count in {condition}",fontsize=11)
    ax.set_ylabel(f"Tfh-cell count in {condition}",fontsize=11)
    ax.grid(False); sns.despine(ax=ax)

    plt.tight_layout()
    output = f"Fig.5/{files[condition]}"
    plt.savefig(output+".pdf",dpi=300,bbox_inches="tight")
    plt.savefig(output+".tiff",dpi=600,bbox_inches="tight")
    plt.show()

In [ ]:
genes = ["SEC11A","SEC23IP","SEC24A","COPA","COPB1","HSPA5","CANX","ERP44","CALR","ALG2","MGAT1","ST8SIA4",
         "B4GALT3","MAN1A1","GANAB","RPN1","UBE2L6","FBXO9","PSMB8","PSMD6","PSMB10","XBP1","IRF4","FOXO1","PRDM1"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(plasma_cells, var_names=genes, groupby="Condition", standard_scale="var", figsize=(10,2))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.tick_params(axis="x", labelsize=12, rotation=90)
ax.tick_params(axis="y", labelsize=12)
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.5/plasma_TFs_expression.pdf", dpi=300, bbox_inches="tight")
plt.show()

<h1>Palantir</h1>

In [ ]:
sc.pp.log1p(Bcell)
sc.pp.highly_variable_genes(Bcell, n_top_genes=1500)

In [ ]:
palantir.utils.run_diffusion_maps(Bcell, n_components=10)
palantir.utils.determine_multiscale_space(Bcell)

In [ ]:
Bcell.obs_names[Bcell.obs["celltype"] == "B_01_Naive_TCL1A_IGHD"][0]
Bcell.obs_names[Bcell.obs["celltype"] == "B_04_Plasma_IGHA1_IGHG1"][0]

In [ ]:
terminal_states = pd.Series(
    ["B_04_Plasma_IGHA1_IGHG1"],
    index=["ACCAAACGTACGCGTC-1-M01"],
)

In [ ]:
palantir.plot.highlight_cells_on_umap(Bcell, terminal_states)
plt.show()

In [ ]:
pr_res = palantir.core.run_palantir(Bcell, Naive, num_waypoints=1000, terminal_states=terminal_states)

In [ ]:
groups = ["HC","MKPP","SKPP"]
vmin, vmax = Bcell.obs["pseudotime"].agg(["min","max"])
fig, axes = plt.subplots(1,3,figsize=(6,2))

for ax, group in zip(axes,groups):
    sub = Bcell[Bcell.obs["Condition"]==group]
    sc.pl.umap(sub, color="pseudotime", ax=ax, show=False, title=group, frameon=False,
               vmin=vmin, vmax=vmax, cmap="Spectral_r")

plt.tight_layout()
plt.savefig("Fig.S11/Bcell_pseudotime_by_condition_umap_Fixed.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df_plot = Bcell.obs[["Condition","pseudotime"]].dropna()

plt.figure(figsize=(6,4.5))
sns.kdeplot(data=df_plot, x="pseudotime", hue="Condition", hue_order=["HC","MKPP","SKPP"],
            palette=["#4DBBD5","#00A087","#E64B35"], fill=True, alpha=0.3, common_norm=False, linewidth=2)

plt.xlabel("palantir_pseudotime", fontsize=12)
plt.ylabel("Density of B cells", fontsize=12)
plt.title("Shift in B cell developmental trajectory", fontsize=14)
plt.tight_layout()
plt.savefig("Fig.5/Bcell_pseudotime_density.pdf", dpi=300)
plt.show()

<h1>Naive and memory</h1>

In [ ]:
target_clusters = [
    'B_01_Naive_TCL1A_IGHD',
    'B_02_iMemory_IGHD_CD27',
    'B_03_Memory_CD27'
]
subset_cells = Bcell[Bcell.obs['celltype'].isin(target_clusters)].copy()

In [ ]:
sc.tl.rank_genes_groups(subset_cells, "Condition", 
                        method="wilcoxon",
                        groups=['MKPP', 'SKPP'],  
                        reference='HC',          
                        corr_method='benjamini-hochberg',  
                        tie_correct=True)   
result = sc.get.rank_genes_groups_df(subset_cells, group=['MKPP', 'SKPP'])

In [ ]:
logfc_threshold, pval_threshold = 1.5, 0.01
results, up_genes = {}, {}

for group in ["MKPP","SKPP"]:
    df = sc.get.rank_genes_groups_df(subset_cells, group=group)
    sig = df[(df["pvals_adj"]<pval_threshold) & (df["logfoldchanges"].abs()>logfc_threshold)]
    results[group] = sig
    up_genes[group] = set(sig.loc[sig["logfoldchanges"]>0,"names"])

common = up_genes["MKPP"] & up_genes["SKPP"]
mkpp_unique = up_genes["MKPP"] - up_genes["SKPP"]
skpp_unique = up_genes["SKPP"] - up_genes["MKPP"]

pd.DataFrame({"gene":sorted(common)}).to_csv("Fig.S13/Bcell_common_upregulated_genes.csv", index=False)
pd.DataFrame({"gene":sorted(mkpp_unique)}).to_csv("Fig.5/Bcell_MKPP_unique_upregulated_genes.csv", index=False)
pd.DataFrame({"gene":sorted(skpp_unique)}).to_csv("Fig.5/Bcell_SKPP_unique_upregulated_genes.csv", index=False)

plt.figure(figsize=(4,3))
venn = venn2_unweighted(subsets=(len(mkpp_unique),len(skpp_unique),len(common)),
                        set_labels=("MKPP vs HC","SKPP vs HC"), set_colors=("#4DBBD5","#E64B35"), alpha=0.9)

plt.title("log2FC > 1.5, adjusted P < 0.01", fontsize=14, fontweight="bold", pad=10)
for text in list(venn.set_labels)+list(venn.subset_labels):
    if text:
        text.set_fontsize(12)
        text.set_fontweight("bold")

plt.tight_layout()
plt.savefig("Fig.5/Bcell_Venn_upregulated_genes.pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

<h1>Result_6</h1>

In [ ]:
raw_adata = adata.raw.to_adata()
Myeloid = raw_adata[raw_adata.obs['celltype_major2']=='Myeloid cells'].copy()

In [ ]:
markers = ["CST3","LYZ","CD14","FCGR3A","CD83","HLA-DPB1","CCL5","C1QA","C1QB","C1QC","CD24","ARHGAP26","S100A8","S100A9","CD1C","CLEC10A","LILRA4","ITM2C","PPBP","PF4"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#4DBBD5","white","#E64B35"], N=100)

fig, ax = plt.subplots(figsize=(6,3.5), dpi=300)
sc.pl.matrixplot(Myeloid, markers, groupby="celltype", cmap=cmap, title="", vmin=0, vmax=2, colorbar_title="Expression", ax=ax)

for a in fig.axes:
    a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
    a.spines["left"].set_linewidth(0.3); a.spines["bottom"].set_linewidth(0.3)
    a.tick_params(axis="both", width=0.3, length=2)

plt.savefig("Fig.S12/Myeloid_dotplot_markers.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
DC_subtypes = [
   "DC_01_mDC_CD1C_CLEC10A"
]
mDC = Myeloid[Myeloid.obs['celltype'].isin(DC_subtypes)]

In [ ]:
signatures = {
    "Phagocytosis": ["ARF6","CDC42","MARCKSL1","RAC2","CFL1","RPS6KB2","PRKCE","MARCKS","PLCG2"],
    "Antigen_presentation": ["LGMN","CIITA","HLA-DMB","RFX5","HLA-DMA","NFYC","CTSL","IFI30","B2M","HLA-E","TAP2","PSME1","PSME2",
                             "HLA-F","HLA-C","HSP90AB1","HSPA8","HLA-DOA","CD74","HLA-DQA2","HLA-DQB1","HLA-DRA","HLA-DRB1",
                             "HLA-DRB5","HLA-DPA1","HLA-DQA1","HLA-DPB1","HSPA4","CALR","HSP90AA1","HLA-A","PDIA3","CTSB","PSME3",
                             "HLA-B","TAP1","CD4"]
}

for score, genes in signatures.items():
    genes = [g for g in genes if g in mDC.var_names]
    sc.tl.score_genes(mDC, gene_list=genes, score_name=score)

In [ ]:
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("HC","SKPP"),("MKPP","SKPP")]

plot_df = mDC.obs.groupby(["Sample","Condition"], observed=True)["Phagocytosis"].mean().dropna().reset_index()
plot_df = plot_df.rename(columns={"Phagocytosis":"Expression"})

groups = [plot_df.loc[plot_df["Condition"]==c,"Expression"] for c in condition_order]
kw_p = kruskal(*groups).pvalue
dunn = sp.posthoc_dunn(plot_df, val_col="Expression", group_col="Condition", p_adjust="bonferroni")
pvals = [dunn.loc[a,b] for a,b in pairs]
stars = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

fig,ax = plt.subplots(figsize=(3,4))
sns.boxplot(data=plot_df,x="Condition",y="Expression",order=condition_order,ax=ax,width=0.5,showfliers=False,linewidth=1.5)
sns.stripplot(data=plot_df,x="Condition",y="Expression",order=condition_order,hue="Condition",palette=colors,
              ax=ax,size=5,alpha=0.6,jitter=0.15,legend=False,zorder=0)

for j,box in enumerate([p for p in ax.get_children() if isinstance(p,mpl.patches.PathPatch)]):
    box.set_facecolor("none"); box.set_edgecolor(colors[j%3]); box.set_linewidth(1.5); box.set_zorder(2)

for line in ax.lines:
    if len(line.get_xdata()):
        j = int(round(np.mean(line.get_xdata())))
        if 0<=j<3: line.set_color(colors[j]); line.set_linewidth(1.5); line.set_zorder(2)

ymin,ymax = plot_df["Expression"].agg(["min","max"])
yrange = ymax-ymin if ymax!=ymin else 1
heights = [ymax+yrange*h for h in [0.15,0.30,0.45]]

for (a,b),star,h in zip(pairs,stars,heights):
    x1,x2 = condition_order.index(a),condition_order.index(b)
    ax.plot([x1,x2],[h,h],lw=1.2,color="black")
    ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=12,fontweight="bold" if star!="ns" else "normal")

ax.set_ylabel("Phagocytosis Score",fontsize=12,fontweight="bold")
ax.set_xlabel("")
ax.set_xticks(range(3),condition_order,rotation=45,ha="center",fontsize=12)
ax.set_ylim(ymin-yrange*0.1,ymax+yrange*0.6)
sns.despine(ax=ax)
ax.spines[["left","bottom"]].set_linewidth(1.2)
ax.tick_params(width=1.2)

plt.tight_layout()
plt.savefig("Fig.6/mDC_Phagocytosis.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("HC","SKPP"),("MKPP","SKPP")]

plot_df = mDC.obs.groupby(["Sample","Condition"], observed=True)["Antigen_presentation"].mean().dropna().reset_index()
plot_df = plot_df.rename(columns={"Antigen_presentation":"Expression"})

groups = [plot_df.loc[plot_df["Condition"]==c,"Expression"] for c in condition_order]
kw_p = kruskal(*groups).pvalue
dunn = sp.posthoc_dunn(plot_df, val_col="Expression", group_col="Condition", p_adjust="bonferroni")
pvals = [dunn.loc[a,b] for a,b in pairs]
stars = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

fig,ax = plt.subplots(figsize=(3,4))
sns.boxplot(data=plot_df,x="Condition",y="Expression",order=condition_order,ax=ax,width=0.5,showfliers=False,linewidth=1.5)
sns.stripplot(data=plot_df,x="Condition",y="Expression",order=condition_order,hue="Condition",palette=colors,
              ax=ax,size=5,alpha=0.6,jitter=0.15,legend=False,zorder=0)

for j,box in enumerate([p for p in ax.get_children() if isinstance(p,mpl.patches.PathPatch)]):
    box.set_facecolor("none"); box.set_edgecolor(colors[j%3]); box.set_linewidth(1.5); box.set_zorder(2)

for line in ax.lines:
    if len(line.get_xdata()):
        j = int(round(np.mean(line.get_xdata())))
        if 0<=j<3: line.set_color(colors[j]); line.set_linewidth(1.5); line.set_zorder(2)

ymin,ymax = plot_df["Expression"].agg(["min","max"])
yrange = ymax-ymin if ymax!=ymin else 1
heights = [ymax+yrange*h for h in [0.15,0.30,0.45]]

for (a,b),star,h in zip(pairs,stars,heights):
    x1,x2 = condition_order.index(a),condition_order.index(b)
    ax.plot([x1,x2],[h,h],lw=1.2,color="black")
    ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=12,fontweight="bold" if star!="ns" else "normal")

ax.set_ylabel("Antigen Presentation Score",fontsize=12,fontweight="bold")
ax.set_xlabel("")
ax.set_xticks(range(3),condition_order,rotation=45,ha="center",fontsize=12)
ax.set_ylim(ymin-yrange*0.1,ymax+yrange*0.6)
sns.despine(ax=ax)
ax.spines[["left","bottom"]].set_linewidth(1.2)
ax.tick_params(width=1.2)

plt.tight_layout()
plt.savefig("Fig.6/mDC_Antigen_presentation.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
genes = ["HLA-DMB","HLA-DMA","HLA-E","HLA-F","HLA-C","HLA-DOA","HLA-DQA2","HLA-DQB1",
         "HLA-DRA","HLA-DRB1","HLA-DRB5","HLA-DPA1","HLA-DQA1","HLA-DPB1","HLA-A","HLA-B"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(mDC, var_names=genes, groupby="Condition", standard_scale="var", figsize=(8,1.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values(): spine.set_linewidth(0.8)
ax.tick_params(axis="both", labelsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.6/mDC_Antigen_presentation_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
markers = ["ARF6","CDC42","MARCKSL1","RAC2","CFL1","RPS6KB2","PRKCE","MARCKS","PLCG2"]
mDC.obs["Condition"] = pd.Categorical(mDC.obs["Condition"], categories=["HC","MKPP","SKPP"], ordered=True)

df = sc.get.obs_df(mDC, keys=["Condition"]+markers)
mean_df = df.groupby("Condition", observed=False)[markers].mean()
plot_df = ((mean_df-mean_df.min())/(mean_df.max()-mean_df.min())).T

cmap = LinearSegmentedColormap.from_list("custom_trend", ["#4DBBD5","#F0F0F0","#E64B35"], N=100)
fig,ax = plt.subplots(figsize=(2.3,2.8), dpi=300)

sns.heatmap(plot_df, cmap=cmap, linewidths=0.2, linecolor="white", ax=ax,
            cbar_kws={"label":"Mean expression","orientation":"vertical","pad":0.03})

ax.set(xlabel="", ylabel="")
ax.tick_params(axis="y", rotation=0, labelsize=10)
ax.tick_params(axis="x", rotation=45, labelsize=10)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig("Fig.6/mDC_phagocytosis_by_condition2.pdf", dpi=300, bbox_inches="tight")
plt.show()

<h1>pySCENIC</h1>

In [ ]:
WORK_DIR = "/home/xiaoquan/scanpy/KP/Results/pySCENIC"

DATABASE_FNAME = os.path.join(WORK_DIR,"hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather")
TFS_FNAME = os.path.join(WORK_DIR,"hs_hgnc_tfs.txt")
MOTIF_ANNOTATIONS_FNAME = os.path.join(WORK_DIR,"motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl")

{f: os.path.exists(f) for f in [DATABASE_FNAME,TFS_FNAME,MOTIF_ANNOTATIONS_FNAME]}

In [ ]:
WORK_DIR = "/home/xiaoquan/scanpy/KP/Results/6-Result6/pySCENIC/mDC"
os.makedirs(WORK_DIR, exist_ok=True)

tf_names = load_tf_names(TFS_FNAME)
dbs = [RankingDatabase(fname=DATABASE_FNAME, name=os.path.splitext(os.path.basename(DATABASE_FNAME))[0])]

sc.pp.filter_genes(mDC, min_cells=10)

X = mDC.layers["counts"]
if sp.issparse(X): X = X.toarray()
ex_matrix = pd.DataFrame(X, index=mDC.obs_names, columns=mDC.var_names)

with ProgressBar():
    adjacencies = grnboost2(expression_data=ex_matrix, tf_names=tf_names, verbose=True)

adjacencies.head()

In [ ]:
ADJ_FNAME = os.path.join(WORK_DIR, "mDC_adjacencies.tsv")
adjacencies.to_csv(ADJ_FNAME, index=False, sep="\t")

In [ ]:
from pyscenic.utils import modules_from_adjacencies
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))

In [ ]:
import os
import glob
import pickle
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
np.object = object

from dask.diagnostics import ProgressBar
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

In [ ]:
MODULES_FNAME = os.path.join(WORK_DIR, "mDC_modules.p")
with open(MODULES_FNAME, "wb") as f:
    pickle.dump(modules, f)

In [ ]:
motifs_df = prune2df(
    dbs,
    modules,
    MOTIF_ANNOTATIONS_FNAME,
    num_workers=24
)

In [ ]:
MOTIFS_FNAME = os.path.join(WORK_DIR, "mDC_motifs.csv")
motifs_df.to_csv(MOTIFS_FNAME, index=False)

In [ ]:
WORK_DIR = "/home/xiaoquan/scanpy/KP/Results/6-Result6/pySCENIC/mDC"
MOTIFS_FNAME = os.path.join(WORK_DIR,"mDC_motifs.csv")
REGULONS_FNAME = os.path.join(WORK_DIR,"mDC_regulons.p")

motifs_df = load_motifs(MOTIFS_FNAME)
regulons = df2regulons(motifs_df)

with open(REGULONS_FNAME,"wb") as f:
    pickle.dump(regulons,f)

len(regulons)

In [ ]:
auc_mtx = aucell(
    ex_matrix,
    regulons,
    num_workers=4
)

print(auc_mtx.shape)
print(auc_mtx.iloc[:5, :5])

In [ ]:
AUC_FNAME = os.path.join(WORK_DIR, "mDC_auc_mtx.csv")
auc_mtx.to_csv(AUC_FNAME)

In [ ]:
auc_mtx = auc_mtx.loc[mDC.obs_names]

mDC.obsm["X_scenic_auc"] = auc_mtx.values
mDC.uns["scenic_regulons"] = auc_mtx.columns.tolist()

In [ ]:
reg_name = auc_mtx.columns[0]
mDC.obs[reg_name] = auc_mtx[reg_name].values
sc.pl.umap(mDC, color=[reg_name, "Condition"])

In [ ]:
for col in auc_mtx.columns:
    mDC.obs[col] = auc_mtx[col].values

In [ ]:
group_mean_auc = mDC.obs.groupby("Condition")[auc_mtx.columns].mean()

In [ ]:
GROUP_MEAN_AUC_FNAME = os.path.join(WORK_DIR, "mDC_group_mean_auc.csv")
group_mean_auc.to_csv(GROUP_MEAN_AUC_FNAME)

In [ ]:
plt.rcdefaults()
plt.rcParams.update({"font.family":"Arial","pdf.fonttype":42})

core_tfs = ["RFXANK(+)","IRF8(+)"]
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("MKPP","SKPP"),("HC","SKPP")]

sample_df = mDC.obs.groupby(["Sample","Condition"], observed=True)[core_tfs].mean().dropna().reset_index()
fig,axes = plt.subplots(1,len(core_tfs),figsize=(2.8*len(core_tfs),4),facecolor="white")
axes = np.atleast_1d(axes)

for i,(ax,tf) in enumerate(zip(axes,core_tfs)):
    sns.swarmplot(data=sample_df,x="Condition",y=tf,order=condition_order,hue="Condition",
                  palette=colors,ax=ax,size=6,alpha=0.6,legend=False,zorder=1)

    for j,g in enumerate(condition_order):
        vals = sample_df.loc[sample_df["Condition"]==g,tf].dropna()
        q25,med,q75 = np.percentile(vals,[25,50,75])
        ax.plot([j-.2,j+.2],[med,med],color="black",lw=2,zorder=10)
        ax.plot([j-.1,j+.1],[q25,q25],color="black",lw=1.2,zorder=10)
        ax.plot([j-.1,j+.1],[q75,q75],color="black",lw=1.2,zorder=10)
        ax.plot([j,j],[q25,q75],color="black",lw=1.2,zorder=9)

    groups = [sample_df.loc[sample_df["Condition"]==g,tf] for g in condition_order]
    kw_p = kruskal(*groups).pvalue
    tmp = sample_df[["Condition",tf]].rename(columns={tf:"AUC"})
    dunn = sp.posthoc_dunn(tmp,val_col="AUC",group_col="Condition",p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in pairs]
    stars = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

    ymin,ymax = sample_df[tf].agg(["min","max"]); yrange = ymax-ymin if ymax!=ymin else 1
    heights = [ymax+yrange*h for h in [0.15,0.30,0.50]]
    for (a,b),star,h in zip(pairs,stars,heights):
        x1,x2 = condition_order.index(a),condition_order.index(b)
        ax.plot([x1,x2],[h,h],color="black",lw=1)
        ax.text((x1+x2)/2,h+0.01*yrange,star,ha="center",va="bottom",fontsize=10)

    ax.set_title(tf,fontsize=12,fontweight="bold",pad=10)
    ax.set_ylabel("Sample Mean Score" if i==0 else "",fontsize=11,fontweight="bold")
    ax.set_xlabel("")
    ax.set_xticks(range(3),condition_order,rotation=45,ha="right")
    ax.set_ylim(ymin-0.1*yrange,ymax+0.65*yrange)
    sns.despine(ax=ax)
    ax.tick_params(direction="in")

plt.tight_layout()
plt.savefig("Fig.6/mDC_Regulons_Final_Corrected.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
count_df = Myeloid.obs.groupby(["Condition","celltype"], observed=False).size().reset_index(name="count")
count_df["Proportion"] = count_df.groupby("Condition")["count"].transform(lambda x: x/x.sum()*100)
pivot_df = count_df.pivot(index="Condition", columns="celltype", values="Proportion").fillna(0)
pivot_df.to_csv("pivot_df.csv")

celltype_order = ["DC_01_mDC_CD1C_CLEC10A","DC_02_pDC_LILRA4_ITM2C","Mono_01_Classical_CD14","Mono_02_Classical_CD83",
                  "Mono_03_Classical_CD83_HLA-DPB1","Mono_04_Intermedate_CD14_CD16","Mono_05_Intermedate_CD83_HLA-DPB1",
                  "Mono_06_Non-classical_CD16_C1QA","Mono_07_MDSC_CD24_ARHGAP26","Mono_08_MDSC_S100A8_S100A9","Mega"]

cell_colors = {"DC_01_mDC_CD1C_CLEC10A":"#F39B7F","DC_02_pDC_LILRA4_ITM2C":"#00A087","Mono_01_Classical_CD14":"#E64B35",
               "Mono_02_Classical_CD83":"#3C5488","Mono_03_Classical_CD83_HLA-DPB1":"#4DBBD5","Mono_04_Intermedate_CD14_CD16":"#8491B4",
               "Mono_05_Intermedate_CD83_HLA-DPB1":"#91D1C2","Mono_06_Non-classical_CD16_C1QA":"#B9C984",
               "Mono_07_MDSC_CD24_ARHGAP26":"#F494BE","Mono_08_MDSC_S100A8_S100A9":"#EDCAE0","Mega":"#C8CADF"}

def draw_flow(ax,x0,x1,y0b,y0t,y1b,y1t,color):
    x = np.linspace(x0,x1,100)
    s = 1/(1+np.exp(-10*(x-(x0+x1)/2)/(x1-x0)))
    ax.fill_between(x,y0b+(y1b-y0b)*s,y0t+(y1t-y0t)*s,color=color,alpha=0.25,edgecolor="none")

fig,ax = plt.subplots(figsize=(4.5,6))
x = np.arange(len(pivot_df)); width = 0.5
bottom = np.zeros(len(pivot_df)); positions = {}

for ct in celltype_order:
    if ct not in pivot_df.columns: continue
    y = pivot_df[ct].to_numpy(); color = cell_colors[ct]
    ax.bar(x,y,bottom=bottom,width=width,color=color,edgecolor="white",linewidth=1.2,label=ct,zorder=3)
    for i,(b,v) in enumerate(zip(bottom,y)): positions.setdefault(i,{})[ct] = (b,b+v)
    bottom += y

for i in range(len(x)-1):
    for ct in celltype_order:
        if ct in positions[i] and ct in positions[i+1]:
            y0b,y0t = positions[i][ct]; y1b,y1t = positions[i+1][ct]
            draw_flow(ax,x[i]+width/2,x[i+1]-width/2,y0b,y0t,y1b,y1t,cell_colors[ct])

ax.set_xticks(x,pivot_df.index,rotation=45,ha="right",fontsize=16)
ax.set_ylabel("Percentage of cells",fontsize=18)
ax.tick_params(axis="y",labelsize=14)
ax.set_ylim(0,100); ax.set_xlim(-0.5,len(x)-0.5)
sns.despine(ax=ax)

handles,labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1],labels[::-1],title="Cell Types",bbox_to_anchor=(1.05,1),loc="upper left",frameon=False,fontsize=15,title_fontsize=18)

plt.tight_layout()
plt.savefig("Fig.S12/Myeloid_celltypes_Proportion_Alluvial.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
monocyte_subtypes = [
    "Mono_01_Classical_CD14",
    "Mono_02_Classical_CD83",
    "Mono_03_Classical_CD83_HLA-DPB1",
    "Mono_04_Intermedate_CD14_CD16",
    "Mono_05_Intermedate_CD83_HLA-DPB1",
    "Mono_06_Non-classical_CD16_C1QA",
    "Mono_07_MDSC_CD24_ARHGAP26",
    "Mono_08_MDSC_S100A8_S100A9"
]
Mono = Myeloid[Myeloid.obs['celltype'].isin(monocyte_subtypes)]

In [ ]:
markers = ["CD14","S100A8","S100A9","HLA-DRA","HLA-DRB1","HLA-DRB5"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(Mono, var_names=markers, groupby="celltype", standard_scale="var", figsize=(5,3.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values(): spine.set_linewidth(0.8)
ax.tick_params(axis="both", labelsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.S12/Mono_HLA-DR.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
monocyte_subtypes = [
    "Mono_07_MDSC_CD24_ARHGAP26",
    "Mono_08_MDSC_S100A8_S100A9"
]
MDSC = Myeloid[Myeloid.obs['celltype'].isin(monocyte_subtypes)]

In [ ]:
genes = ["S100A8","S100A9","CD14","CD74","HLA-DRB5"]
condition_order = ["HC","MKPP","SKPP"]
colors = ["#4DBBD5","#00A087","#E64B35"]
pairs = [("HC","MKPP"),("HC","SKPP"),("MKPP","SKPP")]

expr = MDSC.raw[:,genes].X if MDSC.raw is not None else MDSC[:,genes].X
if hasattr(expr,"toarray"): expr = expr.toarray()

cell_df = pd.DataFrame(expr,index=MDSC.obs_names,columns=genes)
cell_df[["Sample","Condition"]] = MDSC.obs[["Sample","Condition"]]
sample_df = cell_df.groupby(["Sample","Condition"],observed=True)[genes].mean().reset_index()
plot_df = sample_df.melt(id_vars=["Sample","Condition"],value_vars=genes,var_name="Gene",value_name="Expression")

stars_dict = {}
for gene in genes:
    sub = plot_df[plot_df["Gene"]==gene]
    groups = [sub.loc[sub["Condition"]==c,"Expression"] for c in condition_order]
    kw_p = kruskal(*groups).pvalue
    dunn = sp.posthoc_dunn(sub,val_col="Expression",group_col="Condition",p_adjust="bonferroni")
    pvals = [dunn.loc[a,b] for a,b in pairs]
    stars_dict[gene] = ["***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns" for p in pvals]

mpl.rcParams.update({"text.color":"black","axes.labelcolor":"black","xtick.color":"black","ytick.color":"black"})
fig,axes = plt.subplots(1,len(genes),figsize=(8,3.5))

for i,(ax,gene) in enumerate(zip(axes,genes)):
    sub = plot_df[plot_df["Gene"]==gene]
    sns.boxplot(data=sub,x="Condition",y="Expression",order=condition_order,ax=ax,width=0.5,showfliers=False,linewidth=1.5)
    sns.stripplot(data=sub,x="Condition",y="Expression",order=condition_order,hue="Condition",palette=colors,
                  ax=ax,size=4.5,alpha=0.6,jitter=0.15,legend=False)

    for j,box in enumerate([p for p in ax.get_children() if isinstance(p,mpl.patches.PathPatch)]):
        box.set_facecolor("none"); box.set_edgecolor(colors[j%3]); box.set_linewidth(1.5)
    for line in ax.lines:
        if len(line.get_xdata()):
            j = int(round(np.mean(line.get_xdata())))
            if 0<=j<3: line.set_color(colors[j]); line.set_linewidth(1.5)

    ymin,ymax = sub["Expression"].agg(["min","max"]); yrange = ymax-ymin if ymax!=ymin else 1
    heights = [ymax+yrange*h for h in [0.15,0.30,0.45]]
    for (a,b),star,h in zip(pairs,stars_dict[gene],heights):
        x1,x2 = condition_order.index(a),condition_order.index(b)
        ax.plot([x1,x2],[h,h],lw=1.2,color="black")
        ax.text((x1+x2)/2,h,star,ha="center",va="bottom",fontsize=12,fontweight="bold" if star!="ns" else "normal")

    ax.set_title(gene,fontsize=14,fontweight="bold",pad=14)
    ax.set_xlabel("")
    ax.set_ylabel("Expression Level" if i==0 else "",fontsize=12,fontweight="bold")
    ax.set_xticks(range(3),condition_order,rotation=45,ha="center",fontsize=12)
    ax.set_ylim(ymin-yrange*0.1,ymax+yrange*0.6)
    sns.despine(ax=ax)
    ax.spines[["left","bottom"]].set_linewidth(1.2)
    ax.tick_params(width=1.2)

plt.tight_layout(pad=1.5)
plt.savefig("Fig.6/MDSC_HLA_gene_expression_by_condition.pdf",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
markers = ["ARG1","IDO1","STAT3","CD33","SIGLEC9","TGFBR2","TGFB3","ENTPD1","NNMT","TLR5"]
MDSC.obs["Condition"] = pd.Categorical(MDSC.obs["Condition"], categories=["HC","MKPP","SKPP"], ordered=True)

df = sc.get.obs_df(MDSC, keys=["Condition"]+markers)
mean_df = df.groupby("Condition", observed=False)[markers].mean()
plot_df = ((mean_df-mean_df.min())/(mean_df.max()-mean_df.min())).T

cmap = LinearSegmentedColormap.from_list("custom_trend", ["#4DBBD5","#F0F0F0","#E64B35"], N=100)
fig, ax = plt.subplots(figsize=(2.3,3.5), dpi=300)

sns.heatmap(plot_df, cmap=cmap, linewidths=0.2, linecolor="white", ax=ax,
            cbar_kws={"label":"Mean expression","orientation":"vertical","pad":0.03})

ax.set(xlabel="", ylabel="")
ax.tick_params(axis="y", rotation=0, labelsize=10)
ax.tick_params(axis="x", rotation=45, labelsize=10)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
plt.savefig("Fig.6/Mono_immune_regulation.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
monocyte_subtypes = [
    "Mono_01_Classical_CD14",
    "Mono_02_Classical_CD83",
    "Mono_03_Classical_CD83_HLA-DPB1"
]
Classical_Mono = Myeloid[Myeloid.obs['celltype'].isin(monocyte_subtypes)]

In [ ]:
genes = ["S100A8","S100A9","CXCL1","GAS6","MMP9","CTSG","CTSD","HTRA1","HTRA3","SPP1","SOCS1","MKI67",
         "CD86","CCL2","IL15","SMAD6","PELI1","FN1","VEGFB","WNT6","CBL","RPTOR","DOCK9","TET2","ASXL1","ESR1","ALK"]
cmap = LinearSegmentedColormap.from_list("cytotoxic_cmap", ["#4DBBD5","white","#E64B35"])

dp = sc.pl.DotPlot(Classical_Mono, var_names=genes, groupby="Condition", standard_scale="var", figsize=(10,1.5))
dp.style(cmap=cmap, dot_edge_color="black", dot_edge_lw=0.6, grid=True, smallest_dot=20, dot_max=0.4).legend(
    size_title="Pct. exp.", colorbar_title="Avg. exp.")

axes = dp.show(return_axes=True); ax = axes["mainplot_ax"]

for key,title in [("size_legend_ax","Pct. exp."),("color_legend_ax","Avg. exp.")]:
    if key in axes:
        axes[key].tick_params(labelsize=12)
        axes[key].set_title(title, fontsize=12)

for spine in ax.spines.values(): spine.set_linewidth(0.8)
ax.tick_params(axis="both", labelsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set(xlabel="", ylabel="")

plt.tight_layout()
plt.savefig("Fig.6/inflammatory_Immune_markers_by_condition.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sample_col, condition_col, celltype_col = "Sample","Condition","celltype"

target_cd8 = ["CD8T_04_Effector_GZMA_GZMM","CD8T_05_Effector_GZMK","MAIT","γδT"]
scores = ["Exhaustion scores","Apoptosis scores","TCR signaling scores","Cytotoxic scores","Activation effector scores","IFN response scores"]

CD8_exhausted_like = CD8T[CD8T.obs[celltype_col].isin(target_cd8)].copy()
cd8_exhausted_like_score_df = CD8_exhausted_like.obs.groupby(sample_col, observed=True)[scores].mean()
cd8_exhausted_like_score_df.columns = ["CD8_exhausted_like_"+c.replace(" ","_") for c in scores]
cd8_exhausted_like_score_df.to_csv("/home/xiaoquan/scanpy/KP/Results/6-Result6/Fig.6/CD8_exhausted_like_sample_scores.csv")

dfs = [mdsc_gene_sample_df.copy(),cd4_subset_score_df.copy(),cd8_exhausted_like_score_df.copy()]
dfs = [df.set_index(sample_col) if sample_col in df.columns else df for df in dfs]

sample_condition = next(df[condition_col] for df in dfs if condition_col in df.columns)
merged_mmdsc_cd4_cd8_df = dfs[0].drop(columns=condition_col,errors="ignore")

for df in dfs[1:]:
    merged_mmdsc_cd4_cd8_df = merged_mmdsc_cd4_cd8_df.join(df.drop(columns=condition_col,errors="ignore"),how="inner")

merged_mmdsc_cd4_cd8_df[condition_col] = sample_condition.reindex(merged_mmdsc_cd4_cd8_df.index)
merged_mmdsc_cd4_cd8_df.to_csv("/home/xiaoquan/scanpy/KP/Results/6-Result6/Fig.6/mMDSC_CD4_CD8_sample_merged_for_correlation.csv")

merged_mmdsc_cd4_cd8_df.head()

In [ ]:
sample_df = merged_mmdsc_cd4_cd8_df.copy()
x_var = "mMDSC_suppressive_score"
out_dir = "/home/xiaoquan/scanpy/KP/Results/6-Result6/Fig.6"
os.makedirs(out_dir, exist_ok=True)
sample_df = sample_df.loc[:,~sample_df.columns.str.startswith("Unnamed:")]

plot_score_dict = {
    "Pro":["Pro_Proliferation_scores","Pro_Exhaustion_scores","Pro_Treg_signature_scores"],
    "Th1":["Th1_Cytotoxic_scores","Th1_Exhaustion_scores","Th1_Unhelped_CD4_T_scores"],
    "Treg":["Treg_Treg_signature_scores","Treg_Exhaustion_scores","Treg_Activation_effector_scores"],
    "CD8_exhausted_like":["CD8_exhausted_like_Exhaustion_scores","CD8_exhausted_like_Apoptosis_scores","CD8_exhausted_like_Cytotoxic_scores"]
}

label_map = {
    "Pro_Proliferation_scores":"Proliferation","Pro_Exhaustion_scores":"Exhaustion","Pro_Treg_signature_scores":"Treg signature",
    "Th1_Cytotoxic_scores":"Cytotoxic","Th1_Exhaustion_scores":"Exhaustion","Th1_Unhelped_CD4_T_scores":"Unhelped CD4 T",
    "Treg_Treg_signature_scores":"Treg signature","Treg_Exhaustion_scores":"Exhaustion","Treg_Activation_effector_scores":"Activation",
    "CD8_exhausted_like_Exhaustion_scores":"Exhaustion","CD8_exhausted_like_Apoptosis_scores":"Apoptosis","CD8_exhausted_like_Cytotoxic_scores":"Cytotoxic"
}

colors = ["#E64B35","#4DBBD5","#00A087"]

def spearman_bootstrap(df,y,n_boot=5000,seed=42):
    tmp = df[[x_var,y]].dropna()
    x,z = tmp[x_var].to_numpy(float),tmp[y].to_numpy(float)
    rho,p = stats.spearmanr(x,z)
    rng = np.random.default_rng(seed)
    boot = [stats.spearmanr(x[idx],z[idx]).statistic for idx in (rng.integers(0,len(x),len(x)) for _ in range(n_boot))]
    boot = np.asarray(boot); boot = boot[np.isfinite(boot)]
    lo,hi = np.percentile(boot,[2.5,97.5])
    return len(x),rho,lo,hi,p

results = []
for i,(subset,scores) in enumerate(plot_score_dict.items()):
    for j,score in enumerate(scores):
        n,rho,lo,hi,p = spearman_bootstrap(sample_df,score,seed=42+i*3+j)
        results.append([subset,score,label_map[score],n,rho,lo,hi,p])

results_df = pd.DataFrame(results,columns=["Subset","Variable","Functional_score","n","rho","CI_lower","CI_upper","P_raw"])
results_df["P_FDR"] = multipletests(results_df["P_raw"],method="fdr_bh")[1]
results_df["Significance"] = np.select(
    [results_df["P_FDR"]<0.001,results_df["P_FDR"]<0.01,results_df["P_FDR"]<0.05],
    ["***","**","*"],default="ns"
)
results_df.to_csv(f"{out_dir}/mMDSC_T_cell_Spearman_complete_results.csv",index=False)

format_p = lambda p: "<0.001" if p<0.001 else f"{p:.3f}"

formatted_table = results_df[["Subset","Functional_score","n","rho","CI_lower","CI_upper","P_raw","P_FDR","Significance"]].copy()
formatted_table["Spearman rho"] = formatted_table["rho"].map(lambda x:f"{x:.3f}")
formatted_table["95% CI"] = formatted_table.apply(lambda x:f"{x.CI_lower:.3f} to {x.CI_upper:.3f}",axis=1)
formatted_table["Raw P"] = formatted_table["P_raw"].map(format_p)
formatted_table["BH-FDR"] = formatted_table["P_FDR"].map(format_p)
formatted_table = formatted_table[["Subset","Functional_score","n","Spearman rho","95% CI","Raw P","BH-FDR","Significance"]]
formatted_table.to_csv(f"{out_dir}/mMDSC_T_cell_Spearman_formatted_table.csv",index=False)
formatted_table.to_excel(f"{out_dir}/mMDSC_T_cell_Spearman_results.xlsx",index=False)

for subset,scores in plot_score_dict.items():
    fig,ax = plt.subplots(figsize=(4.5,4.2),facecolor="white")

    for i,(score,color) in enumerate(zip(scores,colors)):
        tmp = sample_df[[x_var,score]].dropna()
        stat_row = results_df[(results_df["Subset"]==subset)&(results_df["Variable"]==score)].iloc[0]

        sns.regplot(data=tmp,x=x_var,y=score,ci=95,ax=ax,
                    scatter_kws={"s":45,"color":color,"alpha":0.85,"edgecolors":"white"},
                    line_kws={"color":color,"linewidth":2})

        ax.text(0.04,0.96-i*0.075,
                f"{label_map[score]}: $\\rho$ = {stat_row['rho']:.2f}, FDR = {format_p(stat_row['P_FDR'])}",
                transform=ax.transAxes,ha="left",va="top",fontsize=9.5,color=color,fontweight="bold")

    ax.set_xlabel("mMDSC suppressive score",fontsize=12,fontweight="bold")
    ax.set_ylabel("Functional score",fontsize=12,fontweight="bold")
    ax.set_title(subset.replace("_"," "),fontsize=13,fontweight="bold")
    ax.grid(False); sns.despine(ax=ax)
    ax.tick_params(axis="both",labelsize=10,width=1.1)

    plt.tight_layout()
    output = f"{out_dir}/mMDSC_vs_{subset}_correlation"
    plt.savefig(output+".pdf",dpi=300,bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
CD16_C1QA = Mono[Mono.obs['celltype'] == 'Mono_06_Non-classical_CD16_C1QA'].copy()

In [ ]:
scores = ["C1QA_score","C1QB_score","C1QC_score"]
genes = ["C1QA","C1QB","C1QC"]
colors = ["#E64B35","#4DBBD5","#00A087"]

sbp = adata[adata.obs["Condition"]=="SKPP"].obs
score_df = sbp.groupby("celltype", observed=False)[scores].sum().clip(lower=0)
score_df = score_df.div(score_df.sum(axis=0),axis=1).mul(100)
all_cells = sorted(score_df.index)

fig,axes = plt.subplots(3,1,figsize=(10,12))

for i,(ax,score,gene,color) in enumerate(zip(axes,scores,genes,colors)):
    vals = score_df.loc[all_cells,score]
    bars = ax.bar(range(len(all_cells)),vals,color=color,alpha=0.8,linewidth=0.5)

    ax.set_title(f"{gene} Score",fontsize=14,fontweight="bold")
    ax.set_ylabel("Percentage (%)",fontsize=12,fontweight="bold")
    ax.set_xticks(range(len(all_cells)))
    ax.set_xticklabels(all_cells if i==2 else [],rotation=90,fontsize=8)
    if i==2: ax.set_xlabel("Cell Type",fontsize=12,fontweight="bold")

    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,f"{v:.1f}%",
                ha="center",va="bottom",fontsize=7,fontweight="bold",rotation=90)

    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(0.5)
    ax.grid(False)
    ax.set_xlim(-0.5,len(all_cells)-0.5)
    ax.set_ylim(0,ax.get_ylim()[1]*1.05)

plt.tight_layout(pad=2)
plt.savefig("Fig.S13/C1Q_contribution_separate.pdf",dpi=300,bbox_inches="tight")
plt.show()

score_df.rename(columns=dict(zip(scores,genes))).round(2).to_csv("Fig.S14/C1Q_contribution_SKPP.csv")